In [ ]:
from pathlib import Path
from dataclasses import fields
import importlib.util
import json

import nibabel as nib
import numpy as np

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
SRC = PROJECT_ROOT / 'src' / 'training_v2_smalllesion.py'

# -------- Manual inputs --------
INPUT_MRI_PATH = '/paste/full/path/to/input_t1.nii.gz'
OUTPUT_MASK_PATH = '/paste/full/path/to/output_pred_mask.nii.gz'
RUN_OVERRIDE = None  # optional: '/abs/path/to/runs/<run_id>'
USE_TTA = True
THRESHOLD = None  # None => use config logic (Otsu/decision threshold)
# -------------------------------

INPUT_MRI = Path(INPUT_MRI_PATH).expanduser()
OUTPUT_MASK = Path(OUTPUT_MASK_PATH).expanduser()
if not INPUT_MRI.exists():
    raise SystemExit(f'Input MRI not found: {INPUT_MRI}')
if not SRC.exists():
    raise SystemExit(f'Training module not found: {SRC}')


def _resolve_run() -> Path:
    if RUN_OVERRIDE:
        run = Path(RUN_OVERRIDE).expanduser()
        if not run.is_absolute():
            run = (PROJECT_ROOT / run).resolve()
        return run

    run_dirs = sorted([p for p in (PROJECT_ROOT / 'runs').glob('20*') if p.is_dir()], reverse=True)
    for run in run_dirs:
        if (run / 'models' / 'config.json').exists() and (run / 'callbacks' / 'best_model_dynamic.weights.h5').exists():
            return run
    raise SystemExit('No runnable run found under runs/ with config + best weights')


RUN = _resolve_run()
WEIGHTS = RUN / 'callbacks' / 'best_model_dynamic.weights.h5'
CONFIG = RUN / 'models' / 'config.json'
if not WEIGHTS.exists() or not CONFIG.exists():
    raise SystemExit(f'Run missing config/weights: {RUN}')

spec = importlib.util.spec_from_file_location('seg', SRC)
if spec is None or spec.loader is None:
    raise SystemExit(f'Could not load module spec from {SRC}')
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

cfg_json = json.load(open(CONFIG))
valid_cfg_keys = {f.name for f in fields(seg.DynamicTrainingConfig) if f.init}
cfg_json = {k: v for k, v in cfg_json.items() if k in valid_cfg_keys}
cfg = seg.DynamicTrainingConfig(**cfg_json)
cfg.MODEL_DIR = RUN / 'models'
model = seg.build_model_for_inference(cfg, weights_path=str(WEIGHTS))

img_obj = nib.as_closest_canonical(nib.load(str(INPUT_MRI)))
x = img_obj.get_fdata().astype(np.float32)
patch_size = tuple(cfg.PATCH_SIZE or cfg.INPUT_SHAPE[:-1])

probs = seg.gaussian_tta_predict(
    model,
    x,
    patch_size=patch_size,
    overlap=cfg.GAUSSIAN_TILE_OVERLAP,
    sigma=cfg.GAUSSIAN_TILE_SIGMA,
    tta=USE_TTA,
)

brain_mask = seg.compute_brain_mask(x)
thr = THRESHOLD
if thr is None and not bool(getattr(cfg, 'USE_PER_CASE_OTSU', True)):
    thr = float(getattr(cfg, 'DECISION_THRESHOLD', 0.1))

pred = seg.apply_postprocessing(
    probs,
    threshold=thr,
    min_size=0,
    closing=0,
    brain_mask=brain_mask,
    clamp=getattr(cfg, 'OTSU_CLAMP', (0.05, 0.25)),
    min_prob=float(getattr(cfg, 'OTSU_MIN_PROB', 0.01)),
)
pred_u8 = (pred > 0.5).astype(np.uint8)

OUTPUT_MASK.parent.mkdir(parents=True, exist_ok=True)
nib.save(nib.Nifti1Image(pred_u8, img_obj.affine, img_obj.header), str(OUTPUT_MASK))

print('Run:', RUN)
print('Weights:', WEIGHTS)
print('Input:', INPUT_MRI)
print('Output mask:', OUTPUT_MASK)
print('Prediction stats: p.mean=', float(probs.mean()), ' p.max=', float(probs.max()))

